In [51]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import date, timedelta
import pathlib

# Get the directory where this script is located
BASE_DIR = pathlib.Path().resolve()

# Try to import LightGBM, provide install instructions if it fails
try:
    from lightgbm import LGBMRegressor
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    print("All required libraries imported successfully!")
except ImportError:
    print("Error: lightgbm or scikit-learn package not found.")
    print("Please install them using: pip install lightgbm scikit-learn")

print("Script started...")
print(f"Working directory: {BASE_DIR}")

All required libraries imported successfully!
Script started...
Working directory: /Users/kristianlarsen/Documents/skole/7. semester/MMLIP/MMLIP/MMLIP


## 1. Data Loading Functions

In [ ]:
def load_data():
    """
    Loads all required CSV files and performs initial date conversions.
    """
    print("Loading data...")
    print(f"Looking for files in: {BASE_DIR}")
    try:
        # Build full paths to the files
        receivals_file = BASE_DIR / 'data/kernel/receivals.csv'
        po_file = BASE_DIR / 'data/kernel/purchase_orders.csv'
        materials_file = BASE_DIR / 'data/extended/materials.csv'
        mapping_file = BASE_DIR / 'data/prediction_mapping.csv'
        
        # Load main data using the full paths
        receivals = pd.read_csv(receivals_file)
        purchase_orders = pd.read_csv(po_file)
        
        # Load mapping/metadata
        materials = pd.read_csv(materials_file)
        prediction_mapping = pd.read_csv(mapping_file)

        # Parse dates for filtering first (keep as datetime)
        receivals['date_arrival_dt'] = pd.to_datetime(
            receivals['date_arrival'], utc=True, errors='coerce'
        )
        purchase_orders['created_date_time_dt'] = pd.to_datetime(
            purchase_orders['created_date_time'], utc=True, errors='coerce'
        )
        
        # Now convert to simple date for the rest of the pipeline
        receivals['date_arrival'] = receivals['date_arrival_dt'].dt.date
        purchase_orders['delivery_date'] = pd.to_datetime(
            purchase_orders['delivery_date'], utc=True, errors='coerce'
        ).dt.date
        
        # Prediction Mapping: Convert to simple date
        prediction_mapping['forecast_start_date'] = pd.to_datetime(
            prediction_mapping['forecast_start_date'], errors='coerce'
        ).dt.date
        prediction_mapping['forecast_end_date'] = pd.to_datetime(
            prediction_mapping['forecast_end_date'], errors='coerce'
        ).dt.date
        
        # Drop the temporary datetime columns
        receivals = receivals.drop('date_arrival_dt', axis=1)
        purchase_orders = purchase_orders.drop('created_date_time_dt', axis=1)
        
        # Drop rows where date parsing failed
        receivals = receivals.dropna(subset=['date_arrival', 'rm_id', 'net_weight'])
        purchase_orders = purchase_orders.dropna(subset=['delivery_date', 'product_id', 'quantity'])
        
        return receivals, purchase_orders, materials, prediction_mapping

    except FileNotFoundError as e:
        print(f"Error: File not found: {e.filename}")
        print(f"The script was looking in this directory: {BASE_DIR}")
        print("Please make sure all CSV files are in the correct data subdirectories.")
        return None, None, None, None

In [53]:
def aggregate_daily_data(receivals, purchase_orders, materials):
    """
    Aggregates receivals and purchase orders to a daily level per rm_id.
    """
    print("Aggregating daily data...")
    
    # 1. Aggregate receivals
    daily_receivals = receivals.groupby(
        ['rm_id', 'date_arrival']
    ).net_weight.sum().reset_index()
    
    # 2. Create a map from product_id to rm_id using materials.csv
    rm_product_map = materials[['rm_id', 'product_id']].drop_duplicates().dropna()
    
    # 3. Link POs to rm_id
    po_with_rm = pd.merge(
        purchase_orders,
        rm_product_map,
        on='product_id',
        how='left'
    ).dropna(subset=['rm_id']) # Only keep POs we can map
    
    # 4. Aggregate purchase orders
    daily_po = po_with_rm.groupby(
        ['rm_id', 'delivery_date']
    ).quantity.sum().reset_index()

    # Ensure rm_id is a consistent integer type
    daily_receivals['rm_id'] = daily_receivals['rm_id'].astype(int)
    daily_po['rm_id'] = daily_po['rm_id'].astype(int)
    
    return daily_receivals, daily_po

## 2. Feature Engineering Functions

In [54]:
def create_features_by_rm(df_rm, daily_receivals_rm, daily_po_rm):
    """
    Creates features for a single rm_id's data.
    'df_rm' must have columns: rm_id, forecast_start_date, forecast_end_date
    """
    if df_rm.empty:
        return pd.DataFrame()

    features = df_rm.copy()
    features['original_index'] = features.index
    
    # Date-based features
    features['window_length'] = (
        features['forecast_end_date'] - features['forecast_start_date']
    ).apply(lambda x: x.days + 1)
    
    features['end_month'] = features['forecast_end_date'].apply(lambda x: x.month)
    
    # PO in Window feature
    if not daily_po_rm.empty:
        po_merged = pd.merge(
            features,
            daily_po_rm,
            on='rm_id',
            how='left'
        )
        
        po_in_window_mask = (po_merged['delivery_date'] >= po_merged['forecast_start_date']) & \
                            (po_merged['delivery_date'] <= po_merged['forecast_end_date'])
        
        po_in_window_agg = po_merged[po_in_window_mask].groupby('original_index').quantity.sum().rename('po_in_window')
        
        features = pd.merge(
            features,
            po_in_window_agg,
            left_index=True,
            right_index=True,
            how='left'
        ).fillna({'po_in_window': 0})
    else:
        features['po_in_window'] = 0
        
    # Historical Features (relative to forecast_start_date)
    hist_agg_list = []
    
    for start_date in features['forecast_start_date'].unique():
        ref_date = start_date - timedelta(days=1)
        hist_start_30d = ref_date - timedelta(days=29)

        hist_rec_30d = daily_receivals_rm[
            (daily_receivals_rm['date_arrival'] >= hist_start_30d) &
            (daily_receivals_rm['date_arrival'] <= ref_date)
        ]
        hist_po_30d = daily_po_rm[
            (daily_po_rm['delivery_date'] >= hist_start_30d) &
            (daily_po_rm['delivery_date'] <= ref_date)
        ]
        
        rec_agg_val = hist_rec_30d.net_weight.sum()
        po_agg_val = hist_po_30d.quantity.sum()
        
        hist_agg_list.append({
            'forecast_start_date': start_date,
            'hist_rec_30d': rec_agg_val,
            'hist_po_30d': po_agg_val
        })
        
    if hist_agg_list:
        all_hist_agg = pd.DataFrame(hist_agg_list)
        features = pd.merge(
            features,
            all_hist_agg,
            on='forecast_start_date',
            how='left'
        ).fillna(0)
    else:
        features['hist_rec_30d'] = 0
        features['hist_po_30d'] = 0

    feature_cols = [
        'rm_id', 'window_length', 'end_month',
        'po_in_window', 'hist_rec_30d', 'hist_po_30d'
    ]
    
    final_feature_cols = feature_cols + ['original_index', 'forecast_start_date', 'forecast_end_date']
    
    return features[final_feature_cols]

In [55]:
def create_features(df, daily_receivals, daily_po):
    """
    Creates features for the given dataframe by processing one rm_id at a time.
    """
    print(f"Creating features for {len(df)} rows...")
    
    all_rms = df['rm_id'].unique()
    all_features = []

    for i, rm_id in enumerate(all_rms):
        if (i + 1) % 50 == 0:
            print(f"  Processing rm_id {i+1}/{len(all_rms)}")
            
        df_rm = df[df['rm_id'] == rm_id]
        daily_receivals_rm = daily_receivals[daily_receivals['rm_id'] == rm_id]
        daily_po_rm = daily_po[daily_po['rm_id'] == rm_id]
        
        features_rm = create_features_by_rm(df_rm, daily_receivals_rm, daily_po_rm)
        all_features.append(features_rm)

    if not all_features:
        return pd.DataFrame()

    final_features = pd.concat(all_features)
    
    # Define feature columns and set index back
    feature_cols = [
        'rm_id', 'window_length', 'end_month',
        'po_in_window', 'hist_rec_30d', 'hist_po_30d'
    ]
    
    # Add original index for target creation later
    final_feature_cols = feature_cols + ['original_index', 'forecast_start_date', 'forecast_end_date']
    
    # Keep original index for target alignment
    if 'original_index' in final_features.columns:
        final_features = final_features.set_index('original_index', drop=False)
    
    return final_features

## 3. Training Data Generation

In [56]:
def generate_training_data(daily_receivals, daily_po, years_to_use):
    """
    Generates a training DataFrame by creating cumulative windows
    from historical years, processing one rm_id at a time to save memory.
    """
    print("Generating training data...")
    
    all_rms = daily_receivals['rm_id'].unique()
    X_train_list = []
    y_train_list = []

    for i, rm_id in enumerate(all_rms):
        if (i + 1) % 50 == 0:
            print(f"  Generating training data for rm_id {i+1}/{len(all_rms)}")

        train_windows_rm = []
        for year in years_to_use:
            start_date = date(year, 1, 1)
            for days in range(1, 152): 
                end_date = start_date + timedelta(days=days)
                if end_date >= date(2025, 1, 1):
                    break
                train_windows_rm.append({
                    'rm_id': rm_id,
                    'forecast_start_date': start_date,
                    'forecast_end_date': end_date
                })
        
        if not train_windows_rm:
            continue
            
        train_df_rm = pd.DataFrame(train_windows_rm)
        
        # Create features for this rm_id
        daily_receivals_rm = daily_receivals[daily_receivals['rm_id'] == rm_id]
        daily_po_rm = daily_po[daily_po['rm_id'] == rm_id]
        
        X_train_rm = create_features_by_rm(train_df_rm, daily_receivals_rm, daily_po_rm)
        
        if X_train_rm.empty:
            continue

        # Create targets for this rm_id
        merged_targets = pd.merge(
            X_train_rm,
            daily_receivals_rm,
            on='rm_id',
            how='left'
        )
        
        target_mask = (merged_targets['date_arrival'] >= merged_targets['forecast_start_date']) & \
                      (merged_targets['date_arrival'] <= merged_targets['forecast_end_date'])
        
        y_agg = merged_targets[target_mask].groupby('original_index').net_weight.sum()
        
        # The index of X_train_rm should be original_index to align with y_agg
        X_train_rm = X_train_rm.set_index('original_index')
        
        y_train_rm = pd.Series(0, index=X_train_rm.index, name='target')
        y_train_rm.update(y_agg)
        
        # Append to lists
        X_train_list.append(X_train_rm)
        y_train_list.append(y_train_rm)

    # Concatenate all parts at the end
    print("Concatenating all training data...")
    X_train = pd.concat(X_train_list)
    y_train = pd.concat(y_train_list)
    
    # Drop helper columns from X_train
    feature_cols = [
        'rm_id', 'window_length', 'end_month',
        'po_in_window', 'hist_rec_30d', 'hist_po_30d'
    ]
    
    return X_train[feature_cols], y_train

## 4. Load and Process Data

Now let's load the data and see what we're working with:

In [57]:
# Load and aggregate data
receivals, purchase_orders, materials, prediction_mapping = load_data()

if receivals is not None:
    print(f"Receivals shape: {receivals.shape}")
    print(f"Purchase orders shape: {purchase_orders.shape}")
    print(f"Materials shape: {materials.shape}")
    print(f"Prediction mapping shape: {prediction_mapping.shape}")
    
    # Show sample data
    print("\nReceival data sample:")
    print(receivals.head())
else:
    print("Failed to load data!")

Loading data...
Looking for files in: /Users/kristianlarsen/Documents/skole/7. semester/MMLIP/MMLIP/MMLIP
Receivals shape: (122356, 10)
Purchase orders shape: (29963, 12)
Materials shape: (1218, 6)
Prediction mapping shape: (30450, 4)

Receival data sample:
   rm_id  product_id  purchase_order_id  purchase_order_item_no  \
0  365.0  91900143.0           208545.0                    10.0   
1  365.0  91900143.0           208545.0                    10.0   
2  365.0  91900143.0           208490.0                    10.0   
3  365.0  91900143.0           208490.0                    10.0   
4  379.0  91900296.0           210435.0                    20.0   

   receival_item_no  batch_id date_arrival receival_status  net_weight  \
0                 1       NaN   2004-06-15       Completed     11420.0   
1                 2       NaN   2004-06-15       Completed     13760.0   
2                 1       NaN   2004-06-15       Completed     11281.0   
3                 2       NaN   2004-06-15 

In [58]:
# Aggregate the data to daily level
daily_receivals, daily_po = aggregate_daily_data(receivals, purchase_orders, materials)

print(f"Daily receivals shape: {daily_receivals.shape}")
print(f"Daily purchase orders shape: {daily_po.shape}")

print("\nDaily receivals sample:")
print(daily_receivals.head())

print("\nDaily purchase orders sample:")
print(daily_po.head())

Aggregating daily data...
Daily receivals shape: (41890, 3)
Daily purchase orders shape: (42832, 3)

Daily receivals sample:
   rm_id date_arrival  net_weight
0    342   2004-06-23     24940.0
1    343   2005-03-29     21760.0
2    345   2004-09-01     22780.0
3    346   2004-06-24       820.0
4    346   2004-06-30     21260.0

Daily purchase orders sample:
   rm_id delivery_date  quantity
0    342    2002-12-30   23980.0
1    342    2003-06-29    2440.0
2    342    2003-12-30   52660.0
3    342    2004-04-29  100000.0
4    342    2004-05-30  179000.0


## 5. Generate Training Data

Generate training data using historical years (2005-2024):

In [59]:
# Generate Training Data (use 2005-2024 for training)
X_train, y_train = generate_training_data(
    daily_receivals, daily_po, 
    years_to_use=[2016,2017,2018,2019,2020,2021,2022, 2023, 2024]
)

print(f"Training features shape: {X_train.shape}")
print(f"Training targets shape: {y_train.shape}")

print("\nTraining features sample:")
print(X_train.head())

print(f"\nTarget statistics:")
print(y_train.describe())

Generating training data...
  Generating training data for rm_id 50/203
  Generating training data for rm_id 50/203
  Generating training data for rm_id 100/203
  Generating training data for rm_id 100/203
  Generating training data for rm_id 150/203
  Generating training data for rm_id 150/203
  Generating training data for rm_id 200/203
Concatenating all training data...
Training features shape: (275877, 6)
Training targets shape: (275877,)

Training features sample:
                rm_id  window_length  end_month  po_in_window  hist_rec_30d  \
original_index                                                                
0                 342              2          1           0.0           0.0   
1                 342              3          1           0.0           0.0   
2                 342              4          1           0.0           0.0   
3                 342              5          1           0.0           0.0   
4                 342              6          1     

In [60]:
# Export the training dataset for later analysis if needed
training_data_to_export = X_train.copy()
training_data_to_export['target'] = y_train
training_data_filename = BASE_DIR / 'training_dataset.csv'

# Save with index because the index is used in the feature/target generation
training_data_to_export.to_csv(training_data_filename, index=True) 
print(f"Successfully exported training data to {training_data_filename}")

Successfully exported training data to /Users/kristianlarsen/Documents/skole/7. semester/MMLIP/MMLIP/MMLIP/training_dataset.csv


## 6. Prepare Test Data

In [61]:
# Create Test Data
# Set 'ID' as the index to track rows
test_df_raw = prediction_mapping.set_index('ID')
X_test = create_features(test_df_raw, daily_receivals, daily_po)

# Keep only final feature columns
feature_cols = X_train.columns.tolist()
X_test = X_test[feature_cols]

print(f"Test features shape: {X_test.shape}")
print("\nTest features sample:")
print(X_test.head())

Creating features for 30450 rows...
  Processing rm_id 50/203
  Processing rm_id 50/203
  Processing rm_id 100/203
  Processing rm_id 100/203
  Processing rm_id 150/203
  Processing rm_id 150/203
  Processing rm_id 200/203
Test features shape: (30450, 6)

Test features sample:
                rm_id  window_length  end_month  po_in_window  hist_rec_30d  \
original_index                                                                
1                 365              2          1           0.0           0.0   
2                 365              3          1           0.0           0.0   
3                 365              4          1           0.0           0.0   
4                 365              5          1           0.0           0.0   
5                 365              6          1           0.0           0.0   

                hist_po_30d  
original_index               
1                10365000.0  
2                10365000.0  
3                10365000.0  
4                1

## 7. Train Model with Quantile Regression

Using LightGBM with 0.2 quantile loss and early stopping:

In [62]:
# Create validation set
X_train_full, X_val, y_train_full, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

print(f"Training set shape: {X_train_full.shape}")
print(f"Validation set shape: {X_val.shape}")

# Convert rm_id to 'category' dtype for LightGBM
X_train_full['rm_id'] = X_train_full['rm_id'].astype('category')
X_val['rm_id'] = pd.Categorical(
    X_val['rm_id'],
    categories=X_train_full['rm_id'].cat.categories
)

# Align test categories with train categories
X_test['rm_id'] = pd.Categorical(
    X_test['rm_id'],
    categories=X_train_full['rm_id'].cat.categories
)

Training set shape: (234495, 6)
Validation set shape: (41382, 6)


In [63]:
# Set up the model with quantile objective
model = LGBMRegressor(
    objective='quantile',  # Use quantile loss
    alpha=0.2,             # Set to 0.2 quantile
    random_state=42,
    n_estimators=1000,     # Increase n_estimators for early stopping
    learning_rate=0.05,    # Lower learning rate
    n_jobs=-1,
    categorical_feature=['rm_id']
)

print("Model configured for 0.2 quantile regression")

Model configured for 0.2 quantile regression


In [64]:
# Train the model with early stopping
print("Starting model training...")
model.fit(
    X_train_full, y_train_full,
    eval_set=[(X_val, y_val)],
    eval_metric='quantile',  # Evaluate on the correct metric
    callbacks=[lgb.early_stopping(100, verbose=True)]
)
print("Model training complete!")

Starting model training...
[LightGBM] [Warning] categorical_feature is set=rm_id, categorical_column=0 will be ignored. Current value: categorical_feature=rm_id
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001433 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 895
[LightGBM] [Info] Number of data points in the train set: 234495, number of used features: 6
Training until validation scores don't improve for 100 rounds


/Users/kristianlarsen/anaconda3/lib/python3.11/site-packages/lightgbm/basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(
/Users/kristianlarsen/anaconda3/lib/python3.11/site-packages/lightgbm/basic.py:2159: UserWarning: categorical_feature in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/Users/kristianlarsen/anaconda3/lib/python3.11/site-packages/lightgbm/basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(
/Users/kristianlarsen/anaconda3/lib/python3.11/site-packages/lightgbm/basic.py:2159: UserWarning: categorical_feature in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


Did not meet early stopping. Best iteration is:
[1000]	valid_0's quantile: 6398.49
Model training complete!


## 8. Generate Predictions and Save Submission

In [65]:
# Generate predictions
print("Generating predictions...")
predictions = model.predict(X_test)

# We can't have negative weight
predictions[predictions < 0] = 0

print(f"Predictions generated: {len(predictions)}")
print(f"Prediction statistics:")
print(f"  Min: {predictions.min():.4f}")
print(f"  Max: {predictions.max():.4f}")
print(f"  Mean: {predictions.mean():.4f}")
print(f"  Std: {predictions.std():.4f}")

Generating predictions...
Predictions generated: 30450
Prediction statistics:
  Min: 0.0000
  Max: 3894356.1389
  Mean: 56334.1967
  Std: 312437.2797
Predictions generated: 30450
Prediction statistics:
  Min: 0.0000
  Max: 3894356.1389
  Mean: 56334.1967
  Std: 312437.2797


In [66]:
# Create submission dataframe
submission = pd.DataFrame({
    'ID': X_test.index,
    'predicted_weight': predictions * 0.95  # Apply 0.95 scaling factor
})

# Save the submission file
submission_filename = BASE_DIR / 'submission.csv'
submission.to_csv(submission_filename, index=False)

print("-" * 30)
print(f"Success! Predictions saved to {submission_filename}")
print(f"Submission file has {len(submission)} rows (expected 30450).")
print("\nSubmission sample:")
print(submission.head())
print("-" * 30)

------------------------------
Success! Predictions saved to /Users/kristianlarsen/Documents/skole/7. semester/MMLIP/MMLIP/MMLIP/submission.csv
Submission file has 30450 rows (expected 30450).

Submission sample:
   ID  predicted_weight
0   1               0.0
1   2               0.0
2   3               0.0
3   4               0.0
4   5               0.0
------------------------------


## Summary

The notebook has successfully:
1. ✅ Loaded and processed historical receival and purchase order data
2. ✅ Created features including window length, purchase orders in window, and historical data
3. ✅ Generated training data from 2005-2024 historical windows
4. ✅ Trained a LightGBM model with 0.2 quantile regression
5. ✅ Generated predictions for the test set
6. ✅ Saved results to `submission.csv`

The model uses quantile regression to predict conservative estimates (20th percentile) which should help with the evaluation metric.

## Debug: Check Data Columns

Let's check what columns actually exist in our datasets:

In [67]:
# Let's check what columns exist in our datasets
print("Receivals columns:")
print(receivals.columns.tolist())
print(f"Receivals date_arrival type: {type(receivals['date_arrival'].iloc[0])}")

print("\nPurchase orders columns:")
print(purchase_orders.columns.tolist())
print(f"Purchase orders delivery_date type: {type(purchase_orders['delivery_date'].iloc[0])}")

print("\nSample values:")
print("Receivals date_arrival sample:", receivals['date_arrival'].head())
print("Purchase orders delivery_date sample:", purchase_orders['delivery_date'].head())

Receivals columns:
['rm_id', 'product_id', 'purchase_order_id', 'purchase_order_item_no', 'receival_item_no', 'batch_id', 'date_arrival', 'receival_status', 'net_weight', 'supplier_id']
Receivals date_arrival type: <class 'datetime.date'>

Purchase orders columns:
['purchase_order_id', 'purchase_order_item_no', 'quantity', 'delivery_date', 'product_id', 'product_version', 'created_date_time', 'modified_date_time', 'unit_id', 'unit', 'status_id', 'status']
Purchase orders delivery_date type: <class 'datetime.date'>

Sample values:
Receivals date_arrival sample: 0    2004-06-15
1    2004-06-15
2    2004-06-15
3    2004-06-15
4    2004-06-15
Name: date_arrival, dtype: object
Purchase orders delivery_date sample: 1    2003-05-26
4    2004-10-27
5    2005-03-10
6    2006-03-26
7    2012-07-30
Name: delivery_date, dtype: object
